# 01 — Business Understanding (CRISP-DM)

Primeira etapa do CRISP-DM: entender o **problema de negócio** antes de tocar em dados.
Este notebook não executa modelagem; ele fixa o objetivo, os KPIs e os critérios de
sucesso que guiam todas as etapas seguintes.

> **De-identificação.** O projeto público usa nomenclatura genérica (unidades, produtos e
> bairros anonimizados). O mapeamento real fica no dossiê interno, não versionado publicamente.


## Contexto do negócio

- **Negócio:** confeitaria artesanal com **duas unidades** em Niterói (RJ) — uma em polo
  comercial e uma em bairro residencial — com forte canal de **delivery/encomenda**.
- **Produtos:** linhas de tortas/belos em múltiplos tamanhos, com faixas *Standard*,
  *Premium* e *com Frutas*.
- **Mecânicas promocionais reais:** "item do mês" (desconto sazonal por SKU) e
  "dose dupla" (leve 2, pague 1).
- **Problema:** decidir, mês a mês, a **melhor política de preço e desconto** para
  maximizar receita/lucro sem destruir margem.


## Objetivo de negócio e objetivo de dados

**Objetivo de negócio:** prescrever a política de preço/desconto (por SKU e por época)
que maximiza o resultado, mantendo o teto de enquadramento fiscal (EPP).

**Objetivo de dados:** estimar a demanda (quantidade vendida) como função de preço,
desconto, sazonalidade e contexto, para então **otimizar** a receita esperada.


## O catálogo (de-identificado)

Os preços abaixo são reais; apenas os rótulos foram anonimizados. Eles definem as
faixas de preço que a modelagem precisa respeitar.

In [1]:
import pandas as pd

from src.data.gerar_dataset import CATALOGO

catalogo = pd.DataFrame(
    [
        {
            "produto": p.nome,
            "categoria": p.categoria,
            "preco_original": p.preco_original,
            "custo_estimado": round(p.preco_original * p.custo_ratio, 2),
            "elasticidade": p.elasticidade,
        }
        for p in CATALOGO
    ]
)
print(f"{len(catalogo)} SKUs no catálogo de-identificado")
catalogo.groupby("categoria")["preco_original"].agg(["count", "min", "max"]).round(2)

24 SKUs no catálogo de-identificado


,count,min,max
categoria,,,
Doce,1,240.0,240.0
Kit,2,379.0,389.0
Salgado,4,38.0,189.0
Torta,17,24.0,519.0


## KPIs e critérios de sucesso

| Dimensão | Métrica | Critério |
|---|---|---|
| Negócio | Receita/lucro esperado em holdout | Supera a política vigente |
| Modelo | R² / MAE em holdout (alvo: demanda) | R² melhora a cada fase |
| Robustez | Generalização temporal (treino passado → teste futuro) | Sem vazamento |
| Governança | Reprodutibilidade (SEED=42, DVC) | `dvc repro` idêntico |

## Riscos e mitigação

- **Vazamento de dados:** colunas agregadas do próprio dia-alvo (ex.: vendas do mês)
  jamais podem ser features → *guardrail* explícito na preparação.
- **Fronteira temporal:** treino só vê o passado; avaliação no futuro (Q4 festivo).
- **Overfitting de promoção:** validar sempre fora da amostra, por período.
